In [16]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [17]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [18]:
import json


def generate_dataset():
    prompt = """
プロンプト評価用のデータセットを生成してください。このデータセットは、AWSに関連するタスクに特化したPython、JSON、またはRegexを生成するプロンプトの評価に使用します。各オブジェクトがPython、JSON、またはRegexで完了できるタスクを表すJSONオブジェクトの配列を生成してください。

出力例:
```json
[
    {
        "task": "タスクの説明",
    },
    ...
]
```

* 単一のPython関数、単一のJSONオブジェクト、または正規表現で解決できるタスクに絞ること
* あまり多くのコードを書く必要がないタスクに絞ること

3つのオブジェクトを生成してください。
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")  # プリフィル：JSONだけ返させる
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


dataset = generate_dataset()
print(json.dumps(dataset, indent=2))

[
  {
    "task": "AWS S3\u30d0\u30b1\u30c3\u30c8\u540d\u306e\u691c\u8a3c\u7528\u306e\u6b63\u898f\u8868\u73fe\u3092\u4f5c\u6210\u3057\u3066\u304f\u3060\u3055\u3044\u3002S3\u30d0\u30b1\u30c3\u30c8\u540d\u306f3\u301c63\u6587\u5b57\u3067\u3001\u5c0f\u6587\u5b57\u3001\u6570\u5b57\u3001\u30cf\u30a4\u30d5\u30f3\u306e\u307f\u3092\u542b\u307f\u3001\u5148\u982d\u3068\u672b\u5c3e\u304c\u30cf\u30a4\u30d5\u30f3\u4ee5\u5916\u3067\u3042\u308b\u5fc5\u8981\u304c\u3042\u308a\u307e\u3059\u3002"
  },
  {
    "task": "AWS IAM\u30e6\u30fc\u30b6\u30fc\u306e\u30a2\u30af\u30bb\u30b9\u30ad\u30fcID\u3068\u30b7\u30fc\u30af\u30ec\u30c3\u30c8\u30a2\u30af\u30bb\u30b9\u30ad\u30fc\u306e\u30da\u30a2\u3092\u542b\u3080JSON\u30d5\u30a1\u30a4\u30eb\u306e\u69cb\u9020\u3092\u5b9a\u7fa9\u3057\u3066\u304f\u3060\u3055\u3044\u3002\u30e6\u30fc\u30b6\u30fc\u540d\u3001\u30a2\u30af\u30bb\u30b9\u30ad\u30fcID\u3001\u30b7\u30fc\u30af\u30ec\u30c3\u30c8\u30a2\u30af\u30bb\u30b9\u30ad\u30fc\u3001\u4f5c\u6210\u65e5\u6642\u3092\u542b\u3081\

In [19]:
# データセットをファイルに保存
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

print("dataset.json に保存しました")

dataset.json に保存しました


---

## 6. コア評価パイプラインの構築

3つの関数でパイプラインを構成します：
- `run_prompt` → テストケースをプロンプトとマージしてClaudeに送る
- `run_test_case` → 1件のテストケースを実行して採点する
- `run_eval` → データセット全体を処理する

In [20]:
def run_prompt(test_case):
    """テストケースをプロンプトとマージしてClaudeに送り、出力を返す"""
    prompt = f"""以下のタスクを解いてください:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output


def run_test_case(test_case):
    """1件のテストケースを実行して採点する（スコアは仮で10）"""
    output = run_prompt(test_case)

    # TODO: 採点ロジックは次のSTEPで実装
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
    }


def run_eval(dataset):
    """データセット全体を処理して結果リストを返す"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results


print("評価パイプライン関数の定義完了")

評価パイプライン関数の定義完了


## 7. 評価パイプラインを実行する

In [21]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3\u30d0\u30b1\u30c3\u30c8\u540d\u306e\u691c\u8a3c\u6b63\u898f\u8868\u73fe\n\nAWS S3\u30d0\u30b1\u30c3\u30c8\u540d\u306e\u8981\u4ef6\u306b\u5f93\u3063\u305f\u6b63\u898f\u8868\u73fe\u3092\u4f5c\u6210\u3057\u307e\u3059\u3002\n\n## \u6b63\u898f\u8868\u73fe\n\n```regex\n^[a-z0-9]([a-z0-9-]{1,61}[a-z0-9])?$\n```\n\n## \u8a73\u7d30\u8aac\u660e\n\n```regex\n^                    # \u6587\u5b57\u5217\u306e\u958b\u59cb\n[a-z0-9]            # \u5148\u982d: \u5c0f\u6587\u5b57\u307e\u305f\u306f\u6570\u5b57\uff081\u6587\u5b57\uff09\n(\n  [a-z0-9-]{1,61}   # \u4e2d\u9593: \u5c0f\u6587\u5b57\u3001\u6570\u5b57\u3001\u30cf\u30a4\u30d5\u30f3\uff081\u301c61\u6587\u5b57\uff09\n  [a-z0-9]          # \u672b\u5c3e: \u5c0f\u6587\u5b57\u307e\u305f\u306f\u6570\u5b57\uff081\u6587\u5b57\uff09\n)?                   # \u4e2d\u9593\u30fb\u672b\u5c3e\u90e8\u5206\u306f\u7701\u7565\u53ef\u80fd\uff083\u6587\u5b57\u672a\u6e80\u5bfe\u5fdc\uff09\n$                    # \u6587\u5b57\u5217\u306e\u7d

---

## 8. モデル採点者の実装

採点役のClaudeに「強み・弱み・理由・スコア」をJSON形式で返させます。
スコアだけでなく理由も求めることで、採点が中間値（6点付近）に偏るのを防ぎます。

In [22]:
import json
import re


def grade_by_model(test_case, output):
    """採点役のClaudeに回答を評価させてJSONで返す"""
    eval_prompt = f"""あなたは熟練したコードレビュアーです。このAI生成ソリューションを評価してください。

タスク: {test_case["task"]}
ソリューション: {output}

以下のフィールドを持つJSONオブジェクトとして評価を返してください:
- "strengths": 主な強み1〜3つの配列（テキストのみ、コードスニペット不可）
- "weaknesses": 改善点1〜3つの配列（テキストのみ、コードスニペット不可）
- "reasoning": 簡潔な説明（テキストのみ、コードや特殊文字不可）
- "score": 1〜10の数値

重要: レスポンスにコード、バックスラッシュ、特殊文字を含めないでください。
"""
    messages = []
    add_user_message(messages, eval_prompt)
    response = chat(messages, temperature=0.0)

    def parse_eval_json(text):
        text = text.strip()
        if text.startswith("```"):
            lines = text.splitlines()
            if lines and lines[0].strip().startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip() == "```":
                lines = lines[:-1]
            text = "\n".join(lines).strip()
        start = text.find("{")
        if start == -1:
            raise ValueError(f"JSONオブジェクトが見つかりません: {text[:800]}")
        data, _ = json.JSONDecoder().raw_decode(text[start:])
        return data

    data = parse_eval_json(response)
    if "score" not in data:
        for alt in ("Score", "scores"):
            if alt in data:
                data["score"] = data.pop(alt)
                break
    if "score" not in data:
        raise ValueError(
            f"採点JSONに'score'キーがありません。keys={list(data.keys())!r} / 応答先頭: {response[:400]!r}"
        )
    data.setdefault("strengths", [])
    data.setdefault("weaknesses", [])
    if not isinstance(data.get("reasoning"), str):
        data["reasoning"] = str(data.get("reasoning", ""))
    return data


print("grade_by_model 定義完了")

grade_by_model 定義完了


## 9. 採点ロジックを組み込んで評価パイプラインを更新する

In [25]:
from statistics import mean


def run_test_case(test_case):
    """1件のテストケースを実行して採点する"""
    output = run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }


def run_eval(dataset):
    """データセット全体を処理して平均スコアを表示する"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"平均スコア: {average_score}")
    return results


print("run_test_case / run_eval 更新完了")

run_test_case / run_eval 更新完了


## 10. 採点付きで評価パイプラインを実行する

In [26]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

# 各テストケースのスコアと理由を表示
for i, result in enumerate(results):
    print(f"\n--- テストケース {i+1} ---")
    print(f"タスク  : {result['test_case']['task'][:60]}...")
    print(f"スコア  : {result['score']}/10")
    print(f"理由    : {result['reasoning']}")

平均スコア: 6.666666666666667

--- テストケース 1 ---
タスク  : AWS S3バケット名の検証用の正規表現を作成してください。S3バケット名は3〜63文字で、小文字、数字、ハイフンのみを...
スコア  : 8/10
理由    : このソリューションは技術的に正確で、正規表現の実装は要件を完全に満たしている。複数言語での実装例と充実したテストケースが強みである。ただし、最小文字数に関する矛盾と、設計意図の説明不足が減点要因となる。全体的には実用的で信頼性の高いソリューションである。

--- テストケース 2 ---
タスク  : AWS IAMユーザーのアクセスキーIDとシークレットアクセスキーのペアを含むJSONファイルの構造を定義してください。...
スコア  : 6/10
理由    : ソリューションは技術的には適切な構造定義を提供しており、スキーマ仕様も正確です。しかし、AWS認証情報を含むJSONファイルの取り扱いは極めてセンシティブであり、暗号化、アクセス制御、監査ログなどのセキュリティ対策についての記述が不足しています。また、実装ガイダンスの明確さにも改善の余地があります。

--- テストケース 3 ---
タスク  : CloudWatch Logsのログエントリからタイムスタンプとエラーメッセージを抽出するPython関数を作成してくだ...
スコア  : 6/10
理由    : このソリューションは基本的な要件を満たし、構造化されたコードで読みやすいが、実際のCloudWatch Logs環境での運用を想定するとタイムスタンプ形式の柔軟性やエラー処理の詳細さに課題がある。小規模な用途には適しているが、本番環境での使用には改善が必要である。


---

## 11. コード採点者の実装

モデル採点に加えて、**構文の正しさ**と**形式（コードのみか）**をプログラムで検証します。

- `validate_json` / `validate_python` / `validate_regex` → 構文チェック
- `grade_syntax` → outputがコードのみかどうか + 構文チェックを組み合わせる

In [27]:
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(output, test_case):
    """フォーマット（コードのみか）と構文の正しさを採点する"""
    fmt = test_case.get("format", "")

    # マークダウンや説明文が含まれていたらフォーマット違反で0点
    if "```" in output or output.strip().startswith("#"):
        return 0

    if fmt == "json":
        return validate_json(output)
    elif fmt == "python":
        return validate_python(output)
    elif fmt == "regex":
        return validate_regex(output)

    # format未指定の場合はフォーマットチェックのみ（構文チェックなし）
    return 10


print("コード採点関数の定義完了")

コード採点関数の定義完了


## 12. データセットに format フィールドを追加して再生成する

In [28]:
def generate_dataset():
    prompt = """
プロンプト評価用のデータセットを生成してください。このデータセットは、AWSに関連するタスクに特化したPython、JSON、またはRegexを生成するプロンプトの評価に使用します。各オブジェクトがPython、JSON、またはRegexで完了できるタスクを表すJSONオブジェクトの配列を生成してください。

出力例:
```json
[
  {
    "task": "タスクの説明",
    "format": "python"
  }
]
```

* "format"フィールドは "python"、"json"、"regex" のいずれかであること
* 単一のPython関数、単一のJSONオブジェクト、または単一の正規表現で解決できるタスクに絞ること
* あまり多くのコードを書く必要がないタスクに絞ること

フォーマットごとに1つずつ、合計3つのオブジェクトを生成してください。
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


dataset = generate_dataset()
print(json.dumps(dataset, indent=2))

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)
print("\ndataset.json を更新しました")

[
  {
    "task": "AWS S3\u30d0\u30b1\u30c3\u30c8\u540d\u306e\u4e00\u89a7\u304b\u3089\u3001\u7279\u5b9a\u306e\u30ea\u30fc\u30b8\u30e7\u30f3\uff08\u4f8b\uff1aus-east-1\uff09\u306b\u5c5e\u3059\u308b\u30d0\u30b1\u30c3\u30c8\u3092\u30d5\u30a3\u30eb\u30bf\u30ea\u30f3\u30b0\u3057\u3066\u8f9e\u66f8\u5f62\u5f0f\u3067\u8fd4\u3059",
    "format": "python"
  },
  {
    "task": "AWS EC2\u30a4\u30f3\u30b9\u30bf\u30f3\u30b9\u306e\u8a73\u7d30\u60c5\u5831\uff08InstanceId\u3001State\u3001InstanceType\u3001LaunchTime\uff09\u3092JSON\u30d5\u30a9\u30fc\u30de\u30c3\u30c8\u3067\u69cb\u9020\u5316\u3059\u308b",
    "format": "json"
  },
  {
    "task": "AWS IAM\u30e6\u30fc\u30b6\u30fc\u306eARN\u6587\u5b57\u5217\uff08\u4f8b\uff1aarn:aws:iam::123456789012:user/username\uff09\u304b\u3089\u3001\u30e6\u30fc\u30b6\u30fc\u540d\u3068\u30a2\u30ab\u30a6\u30f3\u30c8ID\u3092\u62bd\u51fa\u3059\u308b",
    "format": "regex"
  }
]

dataset.json を更新しました


## 13. モデルスコアとコードスコアを統合して評価パイプラインを更新する

In [29]:
def run_prompt(test_case):
    """プロンプトv1: シンプルな指示のみ（改善前のベースライン）"""
    prompt = f"""以下のタスクを解いてください:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output


def run_test_case(test_case):
    output = run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    syntax_score = grade_syntax(output, test_case)

    # モデルスコアとコードスコアの平均
    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "model_score": model_score,
        "syntax_score": syntax_score,
        "score": score,
        "reasoning": model_grade["reasoning"],
    }


def run_eval(dataset):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"平均スコア: {average_score:.2f}")
    return results


print("評価パイプライン更新完了")

評価パイプライン更新完了


## 14. ベースライン評価を実行する（改善前のスコアを記録）

In [30]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

for i, result in enumerate(results):
    print(f"\n--- テストケース {i+1} ({result['test_case'].get('format', '?')}) ---")
    print(f"タスク        : {result['test_case']['task'][:60]}...")
    print(f"モデルスコア  : {result['model_score']}/10")
    print(f"構文スコア    : {result['syntax_score']}/10")
    print(f"合計スコア    : {result['score']}/10")
    print(f"理由          : {result['reasoning']}")

平均スコア: 3.17

--- テストケース 1 (python) ---
タスク        : AWS S3バケット名の一覧から、特定のリージョン（例：us-east-1）に属するバケットをフィルタリングして辞書形式...
モデルスコア  : 6/10
構文スコア    : 0/10
合計スコア    : 3.0/10
理由          : ソリューションはAWSの仕様を理解し、基本的な要件を満たしています。型安全性と拡張性も良好です。しかし、大規模環境での実用性に課題があり、エラーハンドリングが不十分で、本番環境での使用には改善が必要です。

--- テストケース 2 (json) ---
タスク        : AWS EC2インスタンスの詳細情報（InstanceId、State、InstanceType、LaunchTime）...
モデルスコア  : 7/10
構文スコア    : 0/10
合計スコア    : 3.5/10
理由          : このソリューションは実用的で段階的な説明が良好ですが、実装の完全性とスケーラビリティの面で改善の余地があります。基本的な要件は満たしていますが、本番環境での運用を想定した詳細な考慮が不十分です。

--- テストケース 3 (regex) ---
タスク        : AWS IAMユーザーのARN文字列（例：arn:aws:iam::123456789012:user/username...
モデルスコア  : 6/10
構文スコア    : 0/10
合計スコア    : 3.0/10
理由          : 全体的には教育的価値が高く、段階的な学習に適した構成である。しかし方法2の危険性、正規表現の不完全性、そして不完全なコード例により、実装品質としては中程度である。方法3は実用的だが、より詳細なバリデーションやAWS公式ライブラリの活用についての言及がない。


---

## 15. プロンプトを改善してスコアを比較する（演習）

v1の問題点：コードのみを返す指示がない → マークダウンや説明文が混じる → 構文スコア0

v2の改善：
- コードのみを返すよう明示
- 説明・コメント・コードブロック禁止を追加
- プリフィルでコード生成を誘導

In [31]:
# セル「11. コード採点者の実装」をスキップしても動くようにする
if "grade_syntax" not in globals():
    import ast
    import json
    import re

    def validate_json(text):
        try:
            json.loads(text.strip())
            return 10
        except json.JSONDecodeError:
            return 0

    def validate_python(text):
        try:
            ast.parse(text.strip())
            return 10
        except SyntaxError:
            return 0

    def validate_regex(text):
        try:
            re.compile(text.strip())
            return 10
        except re.error:
            return 0

    def grade_syntax(output, test_case):
        """フォーマット（コードのみか）と構文の正しさを採点する"""
        fmt = test_case.get("format", "")

        if "```" in output or output.strip().startswith("#"):
            return 0

        if fmt == "json":
            return validate_json(output)
        elif fmt == "python":
            return validate_python(output)
        elif fmt == "regex":
            return validate_regex(output)

        return 10


def run_prompt_v2(test_case):
    """プロンプトv2: コードのみを返すよう明示した改善版"""
    prompt = f"""以下のタスクを解いてください:

{test_case["task"]}

* PythonコードかJSONまたは正規表現のみで回答すること
* コメントや説明は一切追加しないこと
* マークダウンのコードブロックは使用しないこと
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")  # 末尾の \n を削除
    output = chat(messages, stop_sequences=["```"])
    return output.strip()


def run_eval_compare(dataset, prompt_fn, label):
    """指定したプロンプト関数で評価して平均スコアを返す"""
    results = []
    for test_case in dataset:
        output = prompt_fn(test_case)
        model_grade = grade_by_model(test_case, output)
        model_score = model_grade["score"]
        syntax_score = grade_syntax(output, test_case)
        score = (model_score + syntax_score) / 2
        results.append({
            "test_case": test_case,
            "output": output,
            "model_score": model_score,
            "syntax_score": syntax_score,
            "score": score,
        })

    avg = mean([r["score"] for r in results])
    print(f"\n【{label}】平均スコア: {avg:.2f}")
    for i, r in enumerate(results):
        fmt = r["test_case"].get("format", "?")
        print(f"  テストケース{i+1}({fmt}): モデル={r['model_score']} 構文={r['syntax_score']} 合計={r['score']}")
    return avg


with open("dataset.json", "r") as f:
    dataset = json.load(f)

avg_v1 = run_eval_compare(dataset, lambda tc: run_prompt(tc), "v1: 改善前")
avg_v2 = run_eval_compare(dataset, run_prompt_v2, "v2: 改善後")

print(f"\nスコア差分: {avg_v2 - avg_v1:+.2f}")


【v1: 改善前】平均スコア: 3.33
  テストケース1(python): モデル=7 構文=0 合計=3.5
  テストケース2(json): モデル=7 構文=0 合計=3.5
  テストケース3(regex): モデル=6 構文=0 合計=3.0

【v2: 改善後】平均スコア: 6.17
  テストケース1(python): モデル=5 構文=10 合計=7.5
  テストケース2(json): モデル=6 構文=0 合計=3.0
  テストケース3(regex): モデル=6 構文=10 合計=8.0

スコア差分: +2.83


---

## 16. PromptEvaluator クラスの実装

今まで個別に作ってきた機能（データセット生成・実行・LLM採点）を1つのクラスにまとめます。

In [32]:
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks

    def generate_dataset(self, task_description, prompt_inputs_spec, output_file, num_cases=3):
        """テストケースを自動生成してファイルに保存する"""
        inputs_desc = "\n".join([f'- "{k}": {v}' for k, v in prompt_inputs_spec.items()])
        keys = list(prompt_inputs_spec.keys())
        example = {k: f"<{v}>" for k, v in prompt_inputs_spec.items()}

        prompt = f"""以下のタスクを評価するためのテストケースを{num_cases}件生成してください:
"{task_description}"

各テストケースは以下のフィールドを持つJSONオブジェクトであること:
{inputs_desc}

例:
```json
[{json.dumps(example, ensure_ascii=False)}]
```

多様で現実的なテストケースを{num_cases}件、JSON配列として生成してください。
"""
        messages = []
        add_user_message(messages, prompt)
        add_assistant_message(messages, "```json")
        text = chat(messages, stop_sequences=["```"])
        dataset = json.loads(text)

        with open(output_file, "w") as f:
            json.dump(dataset, f, indent=2, ensure_ascii=False)
        print(f"{len(dataset)}件のテストケースを {output_file} に保存しました")
        return dataset

    def _grade(self, task_description, prompt_inputs, output, extra_criteria=""):
        """LLM採点者がスコアと理由を返す"""
        criteria_section = f"\nAdditional criteria:\n{extra_criteria}" if extra_criteria else ""
        eval_prompt = f"""あなたは専門の評価者です。このAI生成レスポンスを採点してください。

タスク: {task_description}
入力: {json.dumps(prompt_inputs, ensure_ascii=False)}
レスポンス: {output}

以下の基準で評価してください:
- 関連性と正確さ
- 完全性{criteria_section}

以下のJSONオブジェクトを返してください:
- "score": 1〜10の整数
- "reasoning": 一文での説明（特殊文字やコード不可）
"""
        messages = []
        add_user_message(messages, eval_prompt)
        response = chat(messages, temperature=0.0)
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
        return {"score": 5, "reasoning": "採点結果のパースに失敗しました"}

    def run_evaluation(self, run_prompt_function, dataset_file, extra_criteria="", task_description=""):
        """データセット全体を評価して結果を返す"""
        with open(dataset_file, "r") as f:
            dataset = json.load(f)

        results = []
        for i, prompt_inputs in enumerate(dataset):
            output = run_prompt_function(prompt_inputs)
            grade = self._grade(task_description, prompt_inputs, output, extra_criteria)
            results.append({
                "prompt_inputs": prompt_inputs,
                "output": output,
                "score": grade["score"],
                "reasoning": grade["reasoning"],
            })
            print(f"テストケース {i+1}: {grade['score']}/10 - {grade['reasoning']}")

        avg = mean([r["score"] for r in results])
        print(f"\n平均スコア: {avg:.2f}/10")
        return results


print("PromptEvaluator クラスの定義完了")

PromptEvaluator クラスの定義完了


## 17. アスリート向け食事プランのベースライン評価

In [33]:
TASK_DESCRIPTION = "アスリート1人分の1日の食事プランをコンパクトかつ簡潔に作成してください"

evaluator = PromptEvaluator(max_concurrent_tasks=3)

# テストデータを生成
dataset = evaluator.generate_dataset(
    task_description=TASK_DESCRIPTION,
    prompt_inputs_spec={
        "height": "アスリートの身長（cm）",
        "weight": "アスリートの体重（kg）",
        "goal": "アスリートの目標",
        "restrictions": "食事制限",
    },
    output_file="meal_dataset.json",
    num_cases=3,
)

print(json.dumps(dataset, indent=2, ensure_ascii=False))

3件のテストケースを meal_dataset.json に保存しました
[
  {
    "height": 180,
    "weight": 75,
    "goal": "筋肉量の増加",
    "restrictions": "なし"
  },
  {
    "height": 165,
    "weight": 58,
    "goal": "体脂肪の低減",
    "restrictions": "乳製品アレルギー"
  },
  {
    "height": 188,
    "weight": 85,
    "goal": "持久力の向上",
    "restrictions": "ベジタリアン"
  }
]


In [34]:
# ベースライン：意図的にシンプルな指示のみのプロンプト
def run_prompt_meal_v1(prompt_inputs):
    prompt = f"""この人は何を食べるべきですか？

- 身長: {prompt_inputs["height"]}
- 体重: {prompt_inputs["weight"]}
- 目標: {prompt_inputs["goal"]}
- 食事制限: {prompt_inputs["restrictions"]}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


results = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v1,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria="""
回答には以下を含めること：
- 1日の総カロリー
- 三大栄養素（タンパク質・脂質・炭水化物）の内訳
- 各食事の具体的な食品・量・タイミング
""",
)

テストケース 1: 6/10 - タンパク質摂取量の具体的な計算と食事例は適切だが、1日の総カロリー数値が明記されていない、三大栄養素の具体的な内訳（グラム数）が不足している、各食事のタイミングと具体的な量が曖昧であり、タスク要件の完全性を満たしていない。
テストケース 2: 5/10 - レスポンスは乳製品アレルギーへの配慮と基本的な栄養原則は適切に示していますが、タスク要件の1日の総カロリー数値、三大栄養素の具体的な内訳、各食事の具体的な食品・量・タイミングが欠落しており、コンパクトな1日の食事プランとしての完全性に大きく欠けています。
テストケース 3: 6/10 - レスポンスはベジタリアン対応と持久力向上の基本方針は適切だが、タスク要件の1日総カロリー、三大栄養素の具体的内訳、各食事の量とタイミングが不足しており、コンパクトさの要求に対して構成が冗長である。

平均スコア: 5.67/10


---

## 18. 改善1: 冒頭の一文を明確・直接的に書き直す

**v1の問題**: 「この人は何を食べるべきですか？」→ 曖昧な質問形式

**v2の改善**: 動作動詞で始め、何を・誰のために・どんな制約で作るかを明示する
- `Generate`（生成する）などの直接的な動詞から始める
- 質問ではなく指示の形にする

In [35]:
def run_prompt_meal_v2(prompt_inputs):
    prompt = f"""アスリートの食事制限に合わせた1日の食事プランを作成してください。

- 身長: {prompt_inputs["height"]}
- 体重: {prompt_inputs["weight"]}
- 目標: {prompt_inputs["goal"]}
- 食事制限: {prompt_inputs["restrictions"]}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


extra_criteria = """
回答には以下を含めること：
- 1日の総カロリー
- 三大栄養素（タンパク質・脂質・炭水化物）の内訳
- 各食事の具体的な食品・量・タイミング
"""

print("=== v1: 改善前 ===")
results_v1 = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v1,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria=extra_criteria,
)

print("\n=== v2: 冒頭を明確・直接的に改善 ===")
results_v2 = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v2,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria=extra_criteria,
)

avg_v1 = mean([r["score"] for r in results_v1])
avg_v2 = mean([r["score"] for r in results_v2])
print(f"\nスコア差分: {avg_v2 - avg_v1:+.2f} ({avg_v1:.2f} → {avg_v2:.2f})")

=== v1: 改善前 ===
テストケース 1: 7/10 - 基本的な栄養情報と食事例が適切に提供されており関連性が高いが、各食事の具体的なカロリー計算と栄養素の詳細な内訳、食事のタイミング（時間帯）が不足しており、完全性の面でやや物足りない。
テストケース 2: 5/10 - レスポンスは乳製品アレルギーへの配慮と基本的な栄養原則は適切に示されているが、タスク要件の『1日の総カロリー』『三大栄養素の具体的な内訳』『各食事の具体的な食品・量・タイミング』が完全に欠落しており、コンパクトな食事プランというより一般的なガイドラインに留まっている。
テストケース 3: 5/10 - レスポンスは栄養素の種類と食事パターンの基本的な枠組みを提供していますが、タスク要件の1日の総カロリー、三大栄養素の具体的な内訳、各食事の具体的な食品量とタイミングが欠落しており、実用的な食事プランとしての完全性に大きく欠けています。

平均スコア: 5.67/10

=== v2: 冒頭を明確・直接的に改善 ===
テストケース 1: 9/10 - 身長180cm体重75kgの筋肉量増加目標に対して、消費カロリー2700kcal、タンパク質150g、炭水化物337g、脂質75gという科学的根拠のある栄養計算に基づき、7食に分けた具体的な食材・量・タイミングを提示し、全ての必須要件を満たしているが、脂質の詳細な合計値が表示されていない点が若干の減点要因である。
テストケース 2: 8/10 - レスポンスは入力条件を適切に反映し、カロリー・三大栄養素の詳細な内訳、具体的な食材・量・タイミングを提供しており、乳製品アレルギーへの配慮も十分だが、夕食のタンパク質欄に「qualities g」という記入漏れがあり、また栄養素の重要なポイント欄が途中で切れているため完全性に若干の欠落がある。
テストケース 3: 9/10 - 入力条件を完全に満たし、ベジタリアン制限下での持久力向上に適した栄養バランス（カロリー2,900kcal、タンパク質175g、炭水化物450g以上）を実現した具体的で実行可能な食事プランであり、タイミングと食品量も明確に記載されているが、微量栄養素（鉄分、ビタミンB12など）への言及がないため満点ではない。

平均スコア: 8.67/10

スコア差分: +3.00 (5.6

---

## 19. 改善2: 出力品質ガイドラインを追加する

冒頭の指示に加えて「含めること・形式・手順」を具体的に列挙することで、さらにスコアを上げます。

**2種類のガイドライン：**
- **出力品質ガイドライン** → 回答に含めるべき要素・形式・長さを指定
- **処理手順** → Claudeが答えにたどり着く前に考えるべきステップを指定

In [36]:
def run_prompt_meal_v3(prompt_inputs):
    prompt = f"""アスリートの食事制限に合わせた1日の食事プランを作成してください。

- 身長: {prompt_inputs["height"]}
- 体重: {prompt_inputs["weight"]}
- 目標: {prompt_inputs["goal"]}
- 食事制限: {prompt_inputs["restrictions"]}

ガイドライン：
1. 1日の総カロリーを正確に記載すること
2. タンパク質・脂質・炭水化物の量（グラム）を明記すること
3. 朝食・昼食・夕食・間食それぞれの時間帯を指定すること
4. 食事制限に合った食品のみを使用すること
5. すべての食材の分量をグラム単位で記載すること
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


print("=== v2: 冒頭改善のみ ===")
results_v2 = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v2,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria=extra_criteria,
)

print("\n=== v3: ガイドライン追加 ===")
results_v3 = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v3,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria=extra_criteria,
)

avg_v2 = mean([r["score"] for r in results_v2])
avg_v3 = mean([r["score"] for r in results_v3])
print(f"\nスコア差分: {avg_v3 - avg_v2:+.2f} ({avg_v2:.2f} → {avg_v3:.2f})")

=== v2: 冒頭改善のみ ===
テストケース 1: 8/10 - このレスポンスは、入力されたアスリートのプロフィールに基づいて、適切なカロリー設定（2,800～3,000kcal）とタンパク質目標（150～165g）を設定し、6つの食事タイミングで具体的な食品・量・調理例を提示している点で優れているが、最後の栄養バランスサマリーテーブルが不完全であること、および脂質と炭水化物の詳細な内訳が明示されていないことが減点要因である。
テストケース 2: 8/10 - レスポンスは入力条件を適切に反映し、具体的な食事内容・栄養計算・乳製品対応が充実しているが、昼食後スナックに誤字（炭水化hamル）があり、乳製品代替食品リストが途中で切れているため完全性に若干の欠落がある。
テストケース 3: 8/10 - このレスポンスは入力条件を適切に反映し、具体的な食品・量・タイミング・栄養価を明記した実用的な食事プランを提供しており、ベジタリアン制限下での持久力向上に必要なタンパク質と炭水化物のバランスも良好だが、昼食の「炭水化物numero90g」という明らかな誤字と、ホエイプロテインがベジタリアン対応かどうかの曖昧性により満点ではない。

平均スコア: 8.00/10

=== v3: ガイドライン追加 ===
テストケース 1: 8/10 - レスポンスは筋肉増加目標に適切な栄養設計（3000kcal、タンパク質150g）と具体的な食事プラン、詳細な栄養情報表を提供しており、要件をほぼ完全に満たしているが、回答が途中で切れており完全性に欠ける点と、夜食や就寝前の栄養補給に関する記載がない点が減点要因である。
テストケース 2: 8/10 - レスポンスは身長165cm・体重58kgのアスリート向けに適切なカロリー設定（約2,000kcal）と栄養配分を提示し、乳製品アレルギー対応も確認できるが、レスポンスが途中で切れており昼食以降の完全な栄養情報と1日の総合計が不明確であるため、完全性の点で減点となる。
テストケース 3: 8/10 - ベジタリアン制限と持久力向上目標に適切に対応し、具体的な食品・量・タイミング・栄養価を明記した高品質な食事プランだが、夜間食が未完成で1日の総栄養素が目標値に達していない点が減点要因である。

平均スコア: 8.00/10

スコア差分

---

## 20. 改善3: XMLタグで構造化する

複数の変数が混在するプロンプトでは、XMLタグでセクションを区切るとClaudeが内容を正確に理解しやすくなります。
シンプルなプロンプトでは効果が薄いケースもありますが、複雑になるほど有効です。

In [37]:
def run_prompt_meal_v4(prompt_inputs):
    prompt = f"""<athlete_information>
- 身長: {prompt_inputs["height"]}
- 体重: {prompt_inputs["weight"]}
- 目標: {prompt_inputs["goal"]}
- 食事制限: {prompt_inputs["restrictions"]}
</athlete_information>

上記のアスリート情報をもとに、1日の食事プランを作成してください。

ガイドライン：
1. 1日の総カロリーを正確に記載すること
2. タンパク質・脂質・炭水化物の量（グラム）を明記すること
3. 朝食・昼食・夕食・間食それぞれの時間帯を指定すること
4. 食事制限に合った食品のみを使用すること
5. すべての食材の分量をグラム単位で記載すること
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages, max_tokens=2000)


# chat関数にmax_tokensを渡せるよう更新
def chat(messages, system=None, temperature=1.0, stop_sequences=None, use_fast_model=False, max_tokens=1000):
    params = {
        "model": fast_model if use_fast_model else model,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text


print("=== v3: ガイドライン追加 ===")
results_v3 = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v3,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria=extra_criteria,
)

print("\n=== v4: XMLタグ追加 + max_tokens=2000 ===")
results_v4 = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v4,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria=extra_criteria,
)

avg_v3 = mean([r["score"] for r in results_v3])
avg_v4 = mean([r["score"] for r in results_v4])
print(f"\nスコア差分: {avg_v4 - avg_v3:+.2f} ({avg_v3:.2f} → {avg_v4:.2f})")

=== v3: ガイドライン追加 ===
テストケース 1: 7/10 - レスポンスは筋肉増加に適切なカロリー設定と栄養バランスを提示し、具体的な食材・分量・タイミングを含む実用的な食事プランを作成しているが、夕食が途中で切れており完全性に欠け、1日の総合栄養素の最終確認表がないため完全性の点で減点される。
テストケース 2: 8/10 - 入力条件を適切に反映した栄養バランスの良い食事プランで、カロリー・PFC内訳・具体的な食材と量を明記し、乳製品アレルギー対応も示されているが、PM間食が途中で切れており完全性に欠ける点が減点要因である。
テストケース 3: 9/10 - 入力条件を完全に満たし、1日の総カロリー3,050kcal、三大栄養素の詳細な内訳、5つの食事タイミングで具体的な食品・量・栄養価を提供しており、ベジタリアン制限と持久力向上目標に適切に対応しているが、夕食の栄養価記載が不完全な点で満点ではない。

平均スコア: 8.00/10

=== v4: XMLタグ追加 + max_tokens=2000 ===
テストケース 1: 7/10 - 筋肉量増加向けの食事プランとして基本的な栄養学的根拠は適切で、具体的な食材・量・タイミング・栄養素内訳も明記されているが、脂質が目標の58%に留まっている点、表の一部に誤字（ressタンパク質、studies---）がある点、およびTDEE計算の根拠が明示されていない点が減点要因である。
テストケース 2: 9/10 - このレスポンスは、入力条件を完全に満たし、1日の総カロリー1,835kcal、タンパク質125.3g、脂質53.1g、炭水化物218.5gの詳細な栄養内訳を提供し、5つの食事タイミングごとに具体的な食品・分量・栄養価を明記し、乳製品アレルギーへの配慮と体脂肪低減目標に対する実践的なアドバイスを含む非常に完成度の高い食事プランであるが、個人の運動強度や基礎代謝量の詳細な計算根拠の明示がやや不足している点で満点ではない。
テストケース 3: 9/10 - このレスポンスは入力条件（身長188cm、体重85kg、持久力向上、ベジタリアン）を完全に満たし、2,800kcalの総カロリー、タンパク質140g、脂質76.3g、炭水化物419.4gの三大栄養素を詳細に記載し、5つの食事タイミングごとに具体

---

## 21. 改善4: マルチショットプロンプト（例を示す）

高スコアだった回答を「理想的な出力例」としてプロンプトに組み込みます。
言葉で説明するより「こういう回答をして」と直接示す方がClaudeに伝わりやすいです。

In [38]:
# v4の評価結果から最高スコアの回答を例として使用する
best_result = max(results_v4, key=lambda r: r["score"])

EXAMPLE_INPUT = "\n".join([f"- {k}: {v}" for k, v in best_result["prompt_inputs"].items()])
EXAMPLE_OUTPUT = best_result["output"]

def run_prompt_meal_v5(prompt_inputs):
    prompt = f"""<athlete_information>
- 身長: {prompt_inputs["height"]}
- 体重: {prompt_inputs["weight"]}
- 目標: {prompt_inputs["goal"]}
- 食事制限: {prompt_inputs["restrictions"]}
</athlete_information>

上記のアスリート情報をもとに、1日の食事プランを作成してください。

ガイドライン：
1. 1日の総カロリーを正確に記載すること
2. タンパク質・脂質・炭水化物の量（グラム）を明記すること
3. 朝食・昼食・夕食・間食それぞれの時間帯を指定すること
4. 食事制限に合った食品のみを使用すること
5. すべての食材の分量をグラム単位で記載すること

以下は理想的な回答の例です。

<example>
<sample_input>
{EXAMPLE_INPUT}
</sample_input>
<ideal_output>
{EXAMPLE_OUTPUT}
</ideal_output>
この例は、食品の選択と量が詳細に記載されており、アスリートの目標と制限に沿った、よく構造化された回答です。
</example>
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages, max_tokens=2000)


print("=== v4: XMLタグ追加 ===")
results_v4 = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v4,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria=extra_criteria,
)

print("\n=== v5: マルチショット（例を追加） ===")
results_v5 = evaluator.run_evaluation(
    run_prompt_function=run_prompt_meal_v5,
    dataset_file="meal_dataset.json",
    task_description=TASK_DESCRIPTION,
    extra_criteria=extra_criteria,
)

avg_v4 = mean([r["score"] for r in results_v4])
avg_v5 = mean([r["score"] for r in results_v5])
print(f"\nスコア差分: {avg_v5 - avg_v4:+.2f} ({avg_v4:.2f} → {avg_v5:.2f})")

=== v4: XMLタグ追加 ===
テストケース 1: 9/10 - このレスポンスは、入力条件に基づいて科学的根拠のある食事プランを提供しており、1日の総カロリー、三大栄養素の詳細な内訳、具体的な食品・分量・タイミングをすべて含み、視覚的にも分かりやすく整理されているが、脂質が目標値を24%超過している点と、タイミングの根拠説明がやや不足している点で満点ではない。
テストケース 2: 9/10 - このレスポンスは、入力条件を完全に満たし、具体的な食材・分量・栄養値を詳細に提示し、乳製品アレルギー対応も適切に行われており、体脂肪低減目標に向けた高タンパク質設定と実行上のポイントも充実しているが、提示カロリー（2,014kcal）が目標（1,800kcal）を超過している点が若干の減点要因である。
テストケース 3: 9/10 - タスク要件をほぼ完全に満たしており、身長188cm体重85kgのベジタリアンアスリート向けに科学的根拠に基づいた2,800kcalの食事プランを提供し、各食事の具体的な食品・量・タイミング・栄養素内訳を詳細に記載し、持久力向上目標に適切な炭水化物4.5g/kg配分とタンパク質1.3g/kg配分を実現しており、唯一の軽微な課題は脂質が目標78gに対し89.9gと若干高い点だが調整方法も提示されている。

平均スコア: 9.00/10

=== v5: マルチショット（例を追加） ===
テストケース 1: 8/10 - レスポンスは入力条件に基づいた適切な栄養目標設定と6食に分けた具体的な食事プランを提供しており、カロリー・三大栄養素の詳細な内訳表も含まれているが、脂質比率が46%と高めであること、タンパク質摂取量の根拠説明が不十分であること、および実装上の完全性に若干の欠落がある点が減点要因である。
テストケース 2: 9/10 - このレスポンスは入力条件を完全に満たし、1日の総カロリー1,769kcal、タンパク質123.1g、脂質50.9g、炭水化物227.9gの詳細な栄養内訳を提供し、5食すべてに具体的な食品・分量・タイミングを明記し、乳製品アレルギー対応も徹底されており、実行性の高い実践的なプランとなっているが、わずかに記述が冗長な部分（注意事項の最後が途中で切れている）があるため満点ではない。
テストケース 3: 9/10